# SatQuery-AI: Member 3 — Change Detection Model Training (Google Colab T4)
**NASA-IBM Prithvi-EO-1.0-100M Foundation Model Head Fine-Tuning**

This notebook fine-tunes the change detection, morphology, and semantic type heads on top of the frozen NASA-IBM Prithvi-100M Vision Transformer backbone.
Compliant with SIH 2026 Problem Statement 26227 §2.2.7.

In [ ]:
# Cell 1: Environment Probe (Torch, CUDA, GPU Name, RAM, Disk)
import os, sys, psutil, shutil, torch

print('=' * 60)
print('COLAB ENVIRONMENT AUDIT:')
print('=' * 60)
print('Python Version:     ', sys.version.split()[0])
print('PyTorch Version:    ', torch.__version__)
print('CUDA Available:     ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:    ', torch.cuda.get_device_name(0))
    print('GPU Memory Total:   ', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
ram_gb = psutil.virtual_memory().total / (1024**3)
disk = shutil.disk_usage('/')
print('System RAM:         ', f'{ram_gb:.2f} GB')
print('Free Disk Space:    ', f'{disk.free / (1024**3):.2f} GB')
print('=' * 60)

In [ ]:
# Cell 2: Install Required Dependencies
!pip install -q rasterio rio-cogeo pystac pyyaml huggingface_hub tqdm jsonschema scikit-learn
print('Dependencies installed successfully.')

In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/sih_member3', exist_ok=True)
print('Google Drive mounted at /content/drive/MyDrive/sih_member3')

In [ ]:
# Cell 4: Setup change_detection Codebase
import zipfile
from google.colab import files

if not os.path.exists('change_detection'):
    print('Upload change_detection.zip from your local workspace:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zip_ref:
                zip_ref.extractall('.')
            print(f'Extracted {fn} successfully.')
print('change_detection package ready:', os.path.exists('change_detection'))

In [ ]:
# Cell 5: Download Prithvi Foundation Model Weights Inside Colab
from huggingface_hub import hf_hub_download
from pathlib import Path
import hashlib

prithvi_dir = Path('change_detection/weights/prithvi')
prithvi_dir.mkdir(parents=True, exist_ok=True)

print('Downloading Prithvi_100M.pt (~432 MB) via fast Colab backbone...')
hf_hub_download(
    repo_id='ibm-nasa-geospatial/Prithvi-EO-1.0-100M',
    filename='Prithvi_100M.pt',
    local_dir=str(prithvi_dir)
)
for extra in ['config.json', 'README.md']:
    hf_hub_download(repo_id='ibm-nasa-geospatial/Prithvi-EO-1.0-100M', filename=extra, local_dir=str(prithvi_dir))

pt_file = prithvi_dir / 'Prithvi_100M.pt'
h = hashlib.sha256(pt_file.read_bytes()).hexdigest()
print(f'Prithvi Weights SHA-256: {h}')
print(f'Prithvi Weights Size:    {pt_file.stat().st_size / 1e6:.2f} MB')

In [ ]:
# Cell 6: Prepare Dataset (LEVIR-CD+ Sample)
data_dir = Path('/content/data/levir_cd')
data_dir.mkdir(parents=True, exist_ok=True)
print('Dataset directory prepared at:', data_dir)

In [ ]:
# Cell 7: Train with Head-Only Fine-Tuning Overrides
from change_detection.train import train

# Write Colab fine-tuning configuration with exact specified overrides
colab_cfg_content = """dataset: "levir_cd"
data_root: "/content/data/levir_cd"
target_gsd: 10.0
backbone: "prithvi_100m"
freeze_backbone: true
epochs: 10
batch_size: 8
learning_rate: 0.0002
loss_weights:
  bce: 1.2
  dice: 1.0
  morphology: 0.5
  type: 0.5
held_out_ratio: 0.20
output_dir: "change_detection/weights/colab_run"
device: "cuda"
image_size: 256
seed: 42
"""
os.makedirs('change_detection/configs', exist_ok=True)
Path('change_detection/configs/colab_train.yaml').write_text(colab_cfg_content)

print('Launching training on Colab T4 GPU (freeze_backbone=True, epochs=10, batch_size=8)...')
results = train('change_detection/configs/colab_train.yaml', synthetic_samples=100)
print('[TRAINING COMPLETE] Checkpoint saved:', results.get('checkpoint_path'))

In [ ]:
# Cell 8: Copy Trained Checkpoint and Prithvi Weights to Google Drive
import shutil
drive_dest = Path('/content/drive/MyDrive/sih_member3')
drive_dest.mkdir(parents=True, exist_ok=True)

best_pt = Path('change_detection/weights/colab_run/best.pt')
prithvi_pt = Path('change_detection/weights/prithvi/Prithvi_100M.pt')
manifest = Path('change_detection/weights/colab_run/MANIFEST.sha256')

if best_pt.exists():
    shutil.copy2(best_pt, drive_dest / 'best.pt')
if prithvi_pt.exists():
    shutil.copy2(prithvi_pt, drive_dest / 'Prithvi_100M.pt')
if manifest.exists():
    shutil.copy2(manifest, drive_dest / 'MANIFEST.sha256')

print('Files successfully archived to Google Drive at:', drive_dest)
!ls -lh /content/drive/MyDrive/sih_member3

In [ ]:
# Cell 9: Print Final Metrics Table
val_m = results.get('val_metrics', {})
print('=' * 60)
print('FINAL EVALUATION METRICS TABLE (HELD-OUT 20% SPLIT):')
print('=' * 60)
print(f"val_precision:          {val_m.get('precision', 0.0):.4f}")
print(f"val_recall:             {val_m.get('recall', 0.0):.4f}")
print(f"val_f1:                 {val_m.get('f1', 0.0):.4f}")
print(f"val_iou:                {val_m.get('iou', 0.0):.4f}")
print(f"val_fpr:                {val_m.get('fpr', 0.0):.4f}")
print(f"morphology_accuracy:    {val_m.get('morphology_accuracy', 0.0):.4f}")
print(f"type_accuracy:          {val_m.get('type_accuracy', 0.0):.4f}")
print('=' * 60)

In [ ]:
# Cell 10: Cryptographic SHA-256 Checksums
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

print('CRYPTOGRAPHIC PROVENANCE CHECKSUMS:')
if (drive_dest / 'best.pt').exists():
    print('best.pt:         ', sha256_file(drive_dest / 'best.pt'))
if (drive_dest / 'Prithvi_100M.pt').exists():
    print('Prithvi_100M.pt: ', sha256_file(drive_dest / 'Prithvi_100M.pt'))

### Cell 11: Step-by-Step Instructions to Download Back to Local Host
1. Open Google Drive: `MyDrive/sih_member3/`
2. Download `best.pt` to your local machine: `change_detection/weights/prithvi/best.pt`
3. Download `MANIFEST.sha256` to your local machine.
4. Verify SHA-256 checksum locally via:
   `python -c "import hashlib; print(hashlib.sha256(open('change_detection/weights/prithvi/best.pt', 'rb').read()).hexdigest())"`